James Bryan M. Francisco

This Python notebook checks if a string is a valid **multiple balanced brackets**. 

A valid string starts and ends with an exclamation point `!`, has balanced brackets between the two exclamation points, and may contain multiple occurrences of the symbol `x`, interspersed within or around the brackets. Valid brackets are: `[]`, `()`, `<>`, `{}`.

Example of a valid string: `!xx[x({xx})[xxx]x]<xxx>x!`

The Push Down Automaton (PDA) shown below checks if a string is a valid **multiple balanced brackets**. Subsequently, the code below simulates the PDA.

![PDA](PDA.png)

**Constant variables** 

These constants will be referenced throughout the implementation of the PDA.

In [244]:
INPUT_ALPHA = set(">}])x!<{[(")
STACK_ALPHA = set("!<{[(")
EMPTY_STACK_SYMBOL = "Z"
EPSILON_SYMBOL = "E"
FINAL_STATES = set([2]) # q2
START_STATE = 0         # q0
BRCKT_CLOSE = {open : close for open, close in ["<>", "{}", "[]", "()"]} # map open brackets to its close bracket counterpart.

**Transition class** and **State class**


The **Transition class** has 3 attributes, similar to how we define *formal* transitions in PDAs:
- `symbol` - input symbol included in `INPUT_ALPHA`.
- `top` - top of the PDA stack when the transition is used. Should be in `STACK_ALPHA`.
- `replaced_top` - string that replaces the top of the stack when the transition is used. If value is an empty string, pop top of the stack.
- `to_state` - state to transition to.

The **State class** has 2 attributes and 2 helper functions:
- `symbol` - the name of the state. ex., `q0`, `q1`, and `q2`.
- `transitions` - a Python dictionary that maps out all the possible transitions from the state. 
    - key-value pair - (input alphabet being read, top of the stack) : Transition object
        - this way, we can map a Transition object with an input alphabet and a stack alphabet.

- `is_valid_transition()` - checks whether an input_alphabet and a stack_alphabet pair maps to a Transition.
- `get_transition()` - returns a Transition object based on an input_alphabet and a stack_alphabet


In [245]:
class Transition:
    def __init__(self, symbol: str, top: str, replaced_top: str, to_state: int):
        self.symbol = symbol
        self.top = top
        self.replaced_top = replaced_top
        self.to_state = to_state
    
    def __repr__(self):
        return f"({self.symbol}, {self.top}, {self.replaced_top})"

class State:
    input_symbols = set()
    def __init__(self, symbol: str, transitions: dict[tuple, Transition]):
        self.symbol = symbol
        self.transitions = transitions
    
    def is_valid_transition(self, input_c, top_c):
        return (input_c, top_c) in self.transitions

    def get_transition(self, input_c, top_c):
        return self.transitions[(input_c, top_c)]


**Instantiation of states**

See bottom-most code block to see all Transitions per State.

In [246]:
states = [
    State(
        symbol="q0",
        transitions={
            ("!", EMPTY_STACK_SYMBOL) : Transition("!", EMPTY_STACK_SYMBOL, "!"+EMPTY_STACK_SYMBOL, 1)
        }
    ),
    State(
        symbol="q1",
        transitions={
            **{("x", top) : Transition("x", top, top, 1) for top in INPUT_ALPHA},
            **{(open_brckt, top) : Transition(open_brckt, top, open_brckt+top, 1) for open_brckt in BRCKT_CLOSE.keys() for top in INPUT_ALPHA},
            **{(close, open) : Transition(close, open, "", 1) for open, close in BRCKT_CLOSE.items()},
            **{("!", "!") : Transition('!', "!", "", 2)}
        } # ** operator unpacks a dictionary. here, we unpacked dictionary comprehensions and combined then into one.
    ),
    State(
        symbol="q2",
        transitions={(EPSILON_SYMBOL, EMPTY_STACK_SYMBOL) : Transition(EPSILON_SYMBOL, EMPTY_STACK_SYMBOL, EMPTY_STACK_SYMBOL, 2)}
    )
]


Global variables for `is_balanced()` and `main1()` output.

In [247]:
ID_data = [] # [(state, remaining string, stack snapshot)]
is_balanced_response = ""

**is_balanced() implementation**

- Initialize a list that will serve as the PDA stack (`stack` variable). Push the `EMPTY_STACK_SYMBOL`.
- Initialize a state index `s_i` that will access `states`.
- Iterate through `input_string`.
    - For the current top of the stack and input alphabet, check if there exist a transition to use.
        - If there is a transition, adjust stack based on Transition object's `replaced_top` value.
        - If none, break out of the iteration.

PDA-specific implementation: when `i==n`.
- When `i==n`, we consider the character being read to be an epsilon character `E`. This way, we consider the only epsilon transition from the PDA, which only `q2` has. 



In [248]:
def is_balanced(input_string):
    global is_balanced_response

    stack = [EMPTY_STACK_SYMBOL]
    s_i = START_STATE              # state (0, 1, 2)
    i = 0                          # input_string index. if i==n, consider an epsilon character.
    
    while i <= len(input_string):
        input_c = (input_string[i] if i < len(input_string) else "E")
        top_c = stack[-1]

        # store such values for ID
        ID_data.append(
            (
                states[s_i].symbol, 
                input_string[i:] if i < len(input_string) else "E", 
                ''.join(reversed(stack))
            )
        )

        if not states[s_i].is_valid_transition(input_c, top_c):
            break
            
        transition = states[s_i].get_transition(input_c, top_c)

        if transition.replaced_top == "":
            stack.pop()
        else:
            stack[-1] = transition.replaced_top[-1] # replace top of the stack with the last char of replaced_top
            # push all remaining chars of replaced_top in reverse order.
            for c in reversed(transition.replaced_top[:-1]):
                stack.append(c)
        
        s_i = transition.to_state
        i += 1

    if s_i in FINAL_STATES and i > len(input_string):
        is_balanced_response = f"{states[s_i].symbol} is a final state.\n{input_string} is valid and has balanced brackets."
        return True
    elif s_i not in FINAL_STATES and i >= len(input_string):
        # done reading the string BUT is not on a final state.
        is_balanced_response = f"Invalid string. {states[s_i].symbol} is not a final state."
    else:
        is_balanced_response = f"Invalid string. Failed at position {i+1}.\nRemaining unprocessed input string: {input_string[i:]}"
    return False
        


**main1() implementation**

In [249]:
def main1():
    global is_balanced_response
    
    file = open("input.txt")
    input_strings = [s.strip() for s in file.readlines()]
    
    for s in input_strings:
        print(f"Processing {s}")
        is_balanced(s)
        for state, rem_string, stack in ID_data:
            print(f"ID: ({state}, {rem_string}, {stack})")
        print(is_balanced_response)
        print()

        ID_data.clear()
        is_balanced_response = ""


**evaluate() implementation**

O(N) recursive implementation.

`eval(start_i, open_brckt)` returns: 
- (1) the # of x's after evaluating `open_brckt` up to its close bracket counterpart, and 
- (2) the index where the close bracket counterpart is.

In [250]:
def evaluate(input_string):
    def eval(start_i, open_brckt):
        x_count = 0
        i = start_i
        while i < len(input_string):
            if input_string[i] == "x":
                x_count += 1
                i += 1
            elif input_string[i] in "<{[(":
                xs, ends_at = eval(i+1, input_string[i])
                x_count += xs
                i = ends_at+1
            elif input_string[i] == '!' or input_string[i] == BRCKT_CLOSE[open_brckt]:
                break
        
        if open_brckt == "<":
            x_count *= 2
        elif open_brckt == "{":
            x_count += 1
        elif open_brckt == "[":
            x_count = 0
        elif open_brckt == "(":
            x_count = max(0, x_count-1)

        return (x_count, i)

    x_count, ends_at = eval(1, "!")
    assert(ends_at == len(input_string)-1) # make sure that we read all characters of input_string
    return x_count

**main2() implementation**

In [251]:
def main2():
    file = open("input.txt")
    input_strings = [s.strip() for s in file.readlines()]
    
    for s in input_strings:
        if is_balanced(s):
            print(f"{s} - Resulting number of x's: {evaluate(s)}")
        else:
            print(f"{s} - Invalid string.")

Run code block below to run `main1()` and `main2()`.

In [252]:
main1()
main2()

Processing !xx[x({xx})[xxx]x]<xxx>x!
ID: (q0, !xx[x({xx})[xxx]x]<xxx>x!, Z)
ID: (q1, xx[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, [x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, ({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, {xx})[xxx]x]<xxx>x!, ([!Z)
ID: (q1, xx})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, x})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, })[xxx]x]<xxx>x!, {([!Z)
ID: (q1, )[xxx]x]<xxx>x!, ([!Z)
ID: (q1, [xxx]x]<xxx>x!, [!Z)
ID: (q1, xxx]x]<xxx>x!, [[!Z)
ID: (q1, xx]x]<xxx>x!, [[!Z)
ID: (q1, x]x]<xxx>x!, [[!Z)
ID: (q1, ]x]<xxx>x!, [[!Z)
ID: (q1, x]<xxx>x!, [!Z)
ID: (q1, ]<xxx>x!, [!Z)
ID: (q1, <xxx>x!, !Z)
ID: (q1, xxx>x!, <!Z)
ID: (q1, xx>x!, <!Z)
ID: (q1, x>x!, <!Z)
ID: (q1, >x!, <!Z)
ID: (q1, x!, !Z)
ID: (q1, !, !Z)
ID: (q2, E, Z)
q2 is a final state.
!xx[x({xx})[xxx]x]<xxx>x! is valid and has balanced brackets.

Processing ![<]>!
ID: (q0, ![<]>!, Z)
ID: (q1, [<]>!, !Z)
ID: (q1, <]>!, [!Z)
ID: (q1, ]>!, <[!Z)
Invalid string. Failed at position 4.

code to test/check contents of `states`.

In [ ]:
# Tests {states} content. outputs all transitions of all states.
for i in range(3):
    print(f"State {i}")
    for k,t in states[i].transitions.items():
        print(k, t)
        assert(k == (t.symbol, t.top))
    print()